## SPACY tokens
This notebook is experimental. I wanted to see if it's possible to create a model only using spacy's linguistic features. This notebook is meant to be combined with other ideas and if used correctly, it can improve most scores by a bit (only marginally for me).

The features included are:
* Named Entity Recognition: people, places mentioned in the text, works of art, etc.
* Parts of Speech: Verbs, Nouns, Propositions
* Tag: The detailed part-of-speech tag.
* Dep: Syntactic dependency, i.e. the relation between tokens.
* is alpha: Is the token an alpha character?
* is stop: Is the token part of a stop list, i.e. the most common words of the language?
* Tense: Is the text taking place in the past, present?
* Aspect: The aspect of a verb is determined by whether the verb expresses a fact, an ongoing action, a completed action, or the end of an ongoing action. 
* Mood: The mood of a verb refers to the manner in which the verb is expressed. Most verbs are indicative and are used to express statements of fact or opinion. The imperative mood is used to give orders and make requests. The interrogative mood asks questions.

![granpa](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQDxv8IbWsa_ilQX_-V5egdS6FbTW9mw8qBFg&usqp=CAU)

In [ ]:
%%capture
import spacy
nlp = spacy.load('../input/en-core-web-sm/en_core_web_sm/en_core_web_sm-3.2.0') 

# Load Libraries and Data

In [ ]:
%%capture
import numpy as np
import pandas as pd
import os, gc, re, warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from cuml.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import pickle
import sys

In [ ]:
dftr = pd.read_csv("/kaggle/input/feedback-prize-english-language-learning/train.csv")
dftr["src"]="train"
dfte = pd.read_csv("/kaggle/input/feedback-prize-english-language-learning/test.csv")
dfte["src"]="test"
dftr.head()

In [ ]:
target_cols = ['cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions',]

# Make 25 Stratified Folds!

In [ ]:
sys.path.append('../input/iterativestratification')
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
FOLDS = 25
skf = MultilabelStratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

for i,(train_index, val_index) in enumerate(skf.split(dftr,dftr[target_cols])):
    dftr.loc[val_index,'FOLD'] = i
dftr.head(3)


In [ ]:
#nlp = spacy.load('../input/en-core-web-sm/en_core_web_sm/en_core_web_sm-3.2.0') 
def spacy_row(text):
    doc = nlp(text)
    d = {"ORG" : 0, "PERSON": 0, "MONEY": 0,"GPE": 0, "LOCATION": 0, "DATE": 0, "TIME": 0, "OTHER": 0,
        "ORDINAL": 0, "CARDINAL": 0, "PERCENT": 0, "LANGUAGE": 0, "LOC": 0, "NORP": 0, "WORK_OF_ART": 0,
        "EVENT": 0, "FAC" : 0, "PRODUCT": 0, "QUANTITY": 0, "LAW": 0, "NUMSENTS": len(list(doc.sents)),
        'SPACE': 0, 'AUX': 0, 'PROPN': 0, 'CCONJ': 0, 'NUM': 0, 'ADV': 0, 'NOUN': 0, 'PUNCT': 0, 'DET': 0, 
         'VERB': 0, 'SCONJ': 0, 'PART': 0, 'ADJ': 0, 'PRON': 0, 'ADP': 0, 'INTJ': 0, 'SYM': 0, 'X': 0,
        "NOUN_CHUNKS": len([chunk.text for chunk in doc.noun_chunks]), 'NN':0, '.':0, 'VBZ':0, 'PRP$':0, 
         'DT':0, 'TO':0, ',':0, 'VBN':0, 'RBR':0, 'JJS':0, 'PRP':0, 'NNP':0, 'WRB':0, 'JJR':0, 'VB':0, 'JJ':0, 
         'IN':0, 'VBP':0, '_SP':0, 'RP':0, '``':0, 'WP':0, 'VBD':0, 'VBG':0, 'WDT':0, 'NNS':0, 'MD':0, "''":0, 
         'RB':0, 'CC':0, 'CD':0, 'POS':0, 'PDT':0, "EX":0, "UH":0, "RBS":0, '-LRB-':0, '-RRB-':0, ':':0, '$':0,
         'FW':0, "HYPH":0, 'NNPS': 0, 'LS':0, 'XX':0, 'NFP':0, 'WP$':0, 'ADD':0, "AFX":0, 'advcl':0,  'cc':0,
         'advmod':0, 'punct':0, 'dobj':0, 'auxpass':0, 'acl':0, 'ROOT':0, 'prt':0, 'amod':0, 'mark':0, 'pcomp':0, 
         'det':0, 'conj':0, 'neg':0, 'acomp':0, 'poss':0, 'pobj':0, 'compound':0, 'nsubjpass':0, 'csubj':0, 
         'attr':0, 'prep':0, 'dative':0, 'nsubj':0, 'xcomp':0, 'aux':0, 'npadvmod':0, 'nummod':0, 'relcl':0, 
         'dep':0, 'ccomp':0, 'appos':0, 'preconj':0, 'expl':0, 'predet':0, 'case':0, 'intj':0, 'mod':0, 
         'parataxis':0, 'quantmod':0, 'nmod':0, 'oprd':0, 'agent':0, 'csubjpass':0, 'meta':0, 'isalpha':0,
         'notisalpha':0, 'stopords':0, 'notstopords':0, 'mood-Ind':0, 'tense-Pres':0, 'tense-Past':0,
         'asp-Perf':0, 'asp-Prog': 0, 'vf-Fin':0, 'vf-Part':0, 'vf-Inf': 0, 'vf-Ger': 0
        }
    #ents
    for ent in doc.ents:
        try:
            d[ent.label_] += 1
        except: 
            pass
    #pos
    for token in doc:
        if token.pos_ in d:
            d[token.pos_] += 1
    #tag
    for token in doc:
        if token.tag_ in d:
            d[token.tag_] += 1
            
    for token in doc:
        if token.is_alpha:
            d['isalpha'] += 1
        else:
            d['notisalpha'] += 1
            
    for token in doc:
        if token.is_stop:
            d['stopords'] += 1
        else:
            d['notstopords'] += 1
            
    for token in doc:
        if token.dep_ in d:
            d[token.dep_] += 1       
            
    mytokens = list()
    for token in doc:
        if token.morph.get("Mood"):
            for tok in token.morph.get("Mood"):
                if 'mood-' + tok in d:
                    d['mood-' + tok] += 1
                else:
                    mytokens.append('mood-' + tok)
    if mytokens:
        print(set(mytokens))
        
    mytokens = list()
    for token in doc:
        if token.morph.get("Tense"):
            for tok in token.morph.get("Tense"):
                if 'tense-' + tok in d:
                    d['tense-' + tok] += 1
                else:
                    mytokens.append('tense-' + tok)
    if mytokens:
        print(set(mytokens))
        
    mytokens = list()
    for token in doc:
        if token.morph.get("Aspect"):
            for tok in token.morph.get("Aspect"):
                if 'asp-' + tok in d:
                    d['asp-' + tok] += 1
                else:
                    mytokens.append('asp-' + tok)
    if mytokens:
        print(set(mytokens))
        
    mytokens = list()
    for token in doc:
        if token.morph.get("VerbForm"):
            for tok in token.morph.get("VerbForm"):
                if 'vf-' + tok in d:
                    d['vf-' + tok] += 1
                else:
                    mytokens.append('vf-' + tok)
    if mytokens:
        print(set(mytokens))
    
    c = pd.DataFrame(pd.Series(d)).T / len(list(doc))
    c['prop-Ind-Verb'] = -1
    c['prop-asp-Perf-Verb'] = -1
    c['prop-asp-Prog-Verb'] = -1
    c['prop-tense-Pres-Verb'] = -1
    c['prop-tense-Past-Verb'] = -1
    c['prop-vf-Fin-Verb'] = -1
    c['prop-vf-Part-Verb'] = -1
    try:
        c['prop-Ind-Verb'] = d['mood-Ind'] / d['VERB']
        c['prop-asp-Perf-Verb'] = d['asp-Perf'] / d['VERB']
        c['prop-asp-Prog-Verb'] = d['asp-Prog'] / d['VERB']
        c['prop-tense-Pres-Verb'] = d['tense-Pres'] / d['VERB']
        c['prop-tense-Past-Verb'] = d['tense-Past'] / d['VERB']
        c['prop-vf-Fin-Verb'] = d['vf-Fin'] / d['VERB']
        c['prop-vf-Part-Verb'] = d['vf-Part'] / d['VERB']
    except:
        pass
    c['lienil'] = len(list(doc))
    return(c)

spacy_tr = pd.concat(list(dftr['full_text'].apply(spacy_row)))
spacy_te = pd.concat(list(dfte['full_text'].apply(spacy_row)))

column_median_spacy = spacy_tr.median()

spacy_tr.fillna(column_median_spacy, inplace = True)
spacy_te.fillna(column_median_spacy, inplace = True)

In [ ]:
scalerbig = MinMaxScaler()
spacy_tr = scalerbig.fit_transform(spacy_tr)
spacy_te = scalerbig.transform(spacy_te)

In [ ]:
dftr.drop(['full_text'], axis = 1, inplace = True)
dfte.drop(['full_text'], axis = 1, inplace = True)
gc.collect()

# Train RAPIDS cuML SVR
Documentation for RAPIDS SVM is [here][1]

[1]: https://docs.rapids.ai/api/cuml/stable/api.html#support-vector-machines

In [ ]:
from cuml.svm import SVR
import cuml
print('RAPIDS version',cuml.__version__)

# OPTUNA

In [ ]:
def comp_score(y_true,y_pred):
    rmse_scores = []
    for i in range(len(target_cols)):
        rmse_scores.append(np.sqrt(mean_squared_error(y_true[:,i],y_pred[:,i])))
    return np.mean(rmse_scores)

In [ ]:
def objective(trial):
    train_x, test_x, train_y, test_y = train_test_split(spacy_tr, dftr[target_cols], test_size=0.2,random_state=42)
    param = {
        'C': trial.suggest_loguniform('C', 0.3, 10.0),
        'degree': trial.suggest_int('degree', 2, 7),
        'gamma': trial.suggest_loguniform('gamma', 1e-3, 0.01),
        'epsilon': trial.suggest_float('epsilon', 0.05, 0.5),
    }
    ev_preds = np.zeros((len(test_x),6))
    for i, t in enumerate(target_cols):
        clf = SVR(**param)
        clf.fit(train_x, train_y[t])
        ev_preds[:,i] = clf.predict(test_x)
    score = comp_score(test_y[target_cols].values, ev_preds)
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)
params_svr = study.best_params

In [ ]:
from sklearn.metrics import mean_squared_error

preds = []
scores = []
def comp_score(y_true,y_pred):
    rmse_scores = []
    for i in range(len(target_cols)):
        rmse_scores.append(np.sqrt(mean_squared_error(y_true[:,i],y_pred[:,i])))
    return np.mean(rmse_scores)

#for fold in tqdm(range(FOLDS),total=FOLDS):
for fold in range(FOLDS):
    print('#'*25)
    print('### Fold',fold+1)
    print('#'*25)
    
    dftr_ = dftr[dftr["FOLD"]!=fold]
    dfev_ = dftr[dftr["FOLD"]==fold]
    
    tr_text_feats = spacy_tr[list(dftr_.index),:]
    ev_text_feats = spacy_tr[list(dfev_.index),:]
    
    ev_preds = np.zeros((len(ev_text_feats),6))
    test_preds = np.zeros((len(spacy_te),6))
    for i,t in enumerate(target_cols):
        print(t,', ',end='')
        clf = SVR(**params_svr)
        clf.fit(tr_text_feats, dftr_[t].values)
        ev_preds[:,i] = clf.predict(ev_text_feats)
        test_preds[:,i] = clf.predict(spacy_te)
    print()
    score = comp_score(dfev_[target_cols].values,ev_preds)
    scores.append(score)
    print("Fold : {} RSME score: {}".format(fold,score))
    preds.append(test_preds)
    
print('#'*25)
print('Overall CV RSME =',np.mean(scores))


# Create Submission CSV

In [ ]:
sub = dfte.copy()

sub.loc[:,target_cols] = np.average(np.array(preds),axis=0) #,weights=[1/s for s in scores]
sub_columns = pd.read_csv("../input/feedback-prize-english-language-learning/sample_submission.csv").columns
sub = sub[sub_columns]


In [ ]:
sub.to_csv("submission.csv",index=None)
sub.head()